# CMIP6 + Observed Heat Story Data Builder: Average Warming → Extreme Hot Days

This Colab notebook prepares CSV files for the DSC 106 final project direction:

> Average warming can feel abstract, so this project translates warming into 35°C+ summer days.

This version keeps the original processing logic and adds observed data:

- Observed average temperature: NOAA nClimGrid monthly `tavg`, 2000–2020
- Observed extreme hot days: NOAA nClimGrid-Daily `tmax`, 2000–2020
- Projected average temperature: CMIP6 monthly `tas`, 2020–2100
- Projected extreme hot days: CMIP6 daily `tasmax`, 2020–2100

It keeps both observed 2020 and CMIP6 2020 so the final CSVs can show observed-baseline deltas, CMIP6-internal deltas, and model-observation baseline gaps.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 0. Install packages

Run this first in Google Colab.

In [2]:
%%capture
!pip install geopandas pyogrio shapely rtree fsspec gcsfs s3fs zarr xarray dask[complete] netCDF4 h5netcdf cftime intake-esm tqdm

## 1. Imports and configuration

Recommended setup for the final project:

- Average temperature: `tas`
- Extreme hot days: daily `tasmax`
- Annual + summer + monthly aggregations
- Summer months: June, July, August
- Scenarios: SSP126, SSP245, SSP585
- Model: GFDL-ESM4

In [3]:
import os
import gc
import json
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import intake
from tqdm.auto import tqdm
import urllib.request

warnings.filterwarnings("ignore")
xr.set_options(display_style="text")

OUTPUT_DIR = Path("/content/cmip6_heat_story_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------
# Time setup
# -----------------------
BASELINE_YEAR = 2020

# Observed history
OBS_START_YEAR = 2000
OBS_END_YEAR = 2020

# CMIP6 projection
# Include 2020 so we can compare CMIP6 2020 with observed 2020.
PROJ_START_YEAR = 2020
PROJ_END_YEAR = 2100

# Keep these names so the old CMIP6 helper functions still work
START_YEAR = PROJ_START_YEAR
END_YEAR = PROJ_END_YEAR
ALL_YEARS = list(range(START_YEAR, END_YEAR + 1))

# Full story range for final D3 files
STORY_YEARS = list(range(OBS_START_YEAR, PROJ_END_YEAR + 1))

# -----------------------
# Seasonal setup
# -----------------------
SUMMER_MONTHS = [6, 7, 8]
SUMMER_LABEL = "JJA"

# -----------------------
# CMIP6 setup
# -----------------------
SOURCE_ID = "GFDL-ESM4"
MEMBER_ID = "r1i1p1f1"
SCENARIOS = ["ssp126", "ssp245", "ssp585"]
SCENARIO_LABELS = {
    "observed": "Observed",
    "ssp126": "Low emissions (SSP126)",
    "ssp245": "Medium emissions (SSP245)",
    "ssp585": "High emissions (SSP585)",
}

# Extreme heat thresholds in Celsius.
# 35°C is more intuitive as "extreme heat"; 30°C is useful if 35°C is too sparse in northern states.
HOT_THRESHOLDS_C = [30, 35]

# Census cartographic boundary files.
COUNTY_ZIP_URL = "https://www2.census.gov/geo/tiger/GENZ2024/shp/cb_2024_us_county_20m.zip"
STATE_ZIP_URL = "https://www2.census.gov/geo/tiger/GENZ2024/shp/cb_2024_us_state_20m.zip"

# Keep 50 states only. Exclude DC and territories.
STATE_ABBRS_50 = {
    "AL","AK","AZ","AR","CA","CO","CT","DE","FL","GA","HI","ID","IL","IN","IA","KS",
    "KY","LA","ME","MD","MA","MI","MN","MS","MO","MT","NE","NV","NH","NJ","NM","NY",
    "NC","ND","OH","OK","OR","PA","RI","SC","SD","TN","TX","UT","VT","VA","WA","WV",
    "WI","WY"
}
# nClimGrid gridded daily and monthly products cover the contiguous U.S. (CONUS).
# Observed rows are therefore built for the lower 48 states.
CONUS_STATE_ABBRS = STATE_ABBRS_50 - {"AK", "HI"}


print("Output directory:", OUTPUT_DIR)
print("CMIP6 years:", START_YEAR, "to", END_YEAR)
print("Observed years:", OBS_START_YEAR, "to", OBS_END_YEAR)
print("Scenarios:", SCENARIOS)
print("Summer months:", SUMMER_MONTHS)
print("Hot thresholds:", HOT_THRESHOLDS_C)

Output directory: /content/cmip6_heat_story_outputs
CMIP6 years: 2020 to 2100
Observed years: 2000 to 2020
Scenarios: ['ssp126', 'ssp245', 'ssp585']
Summer months: [6, 7, 8]
Hot thresholds: [30, 35]


## 2. Load state and county boundaries

This creates county representative points for sampling CMIP6 grid cells.

In [4]:
states = gpd.read_file(STATE_ZIP_URL).to_crs("EPSG:4326")
counties = gpd.read_file(COUNTY_ZIP_URL).to_crs("EPSG:4326")

# Keep only 50 states.
states = states[states["STUSPS"].isin(STATE_ABBRS_50)].copy()

# Join state names/abbreviations onto counties.
state_lookup = states[["STATEFP", "NAME", "STUSPS"]].rename(
    columns={"NAME": "state", "STUSPS": "state_abbr"}
)

counties = counties.merge(state_lookup, on="STATEFP", how="inner")
counties = counties[counties["state_abbr"].isin(STATE_ABBRS_50)].copy()
counties = counties.rename(columns={"NAME": "county"})

# Make sure key columns are strings.
counties["GEOID"] = counties["GEOID"].astype(str).str.zfill(5)
counties["COUNTYFP"] = counties["COUNTYFP"].astype(str).str.zfill(3)
counties["STATEFP"] = counties["STATEFP"].astype(str).str.zfill(2)

county_meta = counties[
    ["GEOID", "STATEFP", "COUNTYFP", "county", "state", "state_abbr", "ALAND", "geometry"]
].copy()

# Representative point is safer than centroid because it stays inside the polygon.
rep_points = county_meta.geometry.representative_point()
county_meta["rep_lon"] = rep_points.x
county_meta["rep_lat"] = rep_points.y

county_attrs = county_meta[["GEOID", "county", "state", "state_abbr", "ALAND"]].copy()

print("States:", len(states))
print("Counties:", len(county_meta))
display(county_meta[["GEOID", "county", "state", "state_abbr", "rep_lon", "rep_lat"]].head())

States: 50
Counties: 3143


,GEOID,county,state,state_abbr,rep_lon,rep_lat
0,18087,LaGrange,Indiana,IN,-85.426353,41.642837
1,20107,Linn,Kansas,KS,-94.843673,38.148909
2,24029,Kent,Maryland,MD,-76.145870,39.188014
3,31065,Furnas,Nebraska,NE,-99.912526,40.175899
4,13237,Putnam,Georgia,GA,-83.346741,33.332538


## 3. Open the CMIP6 catalog and check availability

In [5]:
CATALOG_URL = "https://storage.googleapis.com/cmip6/pangeo-cmip6.json"
cat = intake.open_esm_datastore(CATALOG_URL)

print("Catalog rows:", len(cat.df))
display(cat.df.head())

Catalog rows: 514818


,activity_id,institution_id,source_id,experiment_id,member_id,table_id,variable_id,grid_label,zstore,dcpp_init_year,version
0,HighResMIP,CMCC,CMCC-CM2-HR4,highresSST-present,r1i1p1f1,Amon,ps,gn,gs://cmip6/CMIP6/HighResMIP/CMCC/CMCC-CM2-HR4/...,<NA>,20170706
1,HighResMIP,CMCC,CMCC-CM2-HR4,highresSST-present,r1i1p1f1,Amon,rsds,gn,gs://cmip6/CMIP6/HighResMIP/CMCC/CMCC-CM2-HR4/...,<NA>,20170706
2,HighResMIP,CMCC,CMCC-CM2-HR4,highresSST-present,r1i1p1f1,Amon,rlus,gn,gs://cmip6/CMIP6/HighResMIP/CMCC/CMCC-CM2-HR4/...,<NA>,20170706
3,HighResMIP,CMCC,CMCC-CM2-HR4,highresSST-present,r1i1p1f1,Amon,rlds,gn,gs://cmip6/CMIP6/HighResMIP/CMCC/CMCC-CM2-HR4/...,<NA>,20170706
4,HighResMIP,CMCC,CMCC-CM2-HR4,highresSST-present,r1i1p1f1,Amon,psl,gn,gs://cmip6/CMIP6/HighResMIP/CMCC/CMCC-CM2-HR4/...,<NA>,20170706


In [6]:
# Check whether the needed datasets exist.
REQUIRED_DATASETS = [
    ("tas", "Amon"),
    ("tasmax", "day"),
]

availability_rows = []

for scenario in SCENARIOS:
    for variable_id, table_id in REQUIRED_DATASETS:
        search = cat.search(
            source_id=SOURCE_ID,
            experiment_id=scenario,
            table_id=table_id,
            variable_id=variable_id,
            member_id=MEMBER_ID,
        )
        availability_rows.append({
            "scenario": scenario,
            "variable_id": variable_id,
            "table_id": table_id,
            "matches": len(search.df),
        })

availability = pd.DataFrame(availability_rows)
display(availability)

if (availability["matches"] == 0).any():
    raise ValueError("At least one required CMIP6 dataset is missing. Check SOURCE_ID, MEMBER_ID, scenarios, or variables.")

,scenario,variable_id,table_id,matches
0,ssp126,tas,Amon,1
1,ssp126,tasmax,day,1
2,ssp245,tas,Amon,1
3,ssp245,tasmax,day,1
4,ssp585,tas,Amon,1
5,ssp585,tasmax,day,1


## 4. Helper functions

In [7]:
def _get_coord_name(ds, possible_names):
    for name in possible_names:
        if name in ds.coords or name in ds.dims:
            return name
    raise ValueError(
        f"Could not find coordinate among {possible_names}. "
        f"Available coords: {list(ds.coords)}"
    )


def open_cmip6_variable(scenario, variable_id, table_id):
    # Open one CMIP6 variable for one scenario and subset to 2020-2100.
    search = cat.search(
        source_id=SOURCE_ID,
        experiment_id=scenario,
        table_id=table_id,
        variable_id=variable_id,
        member_id=MEMBER_ID,
    )

    if len(search.df) == 0:
        raise ValueError(
            f"No CMIP6 catalog entries found for "
            f"scenario={scenario}, variable={variable_id}, table={table_id}"
        )

    print(
        f"Found {len(search.df)} catalog entries for "
        f"{scenario}, {variable_id}, {table_id}. Using first matching dataset."
    )

    dsets = search.to_dataset_dict(
        zarr_kwargs={"consolidated": True, "use_cftime": True},
        storage_options={"token": "anon"},
        progressbar=True,
    )

    ds = list(dsets.values())[0]

    lat_name = _get_coord_name(ds, ["lat", "latitude", "y"])
    lon_name = _get_coord_name(ds, ["lon", "longitude", "x"])

    da = ds[variable_id]
    da = da.sel(time=slice(f"{START_YEAR}-01-01", f"{END_YEAR}-12-31"))

    return da, lat_name, lon_name


def to_celsius(da):
    # CMIP6 temperature variables are usually in Kelvin; convert to Celsius.
    return da - 273.15


def sample_county_timeseries(da, lat_name, lon_name, county_meta):
    # Sample an xarray DataArray at county representative points using nearest grid cell.
    lon_values = da[lon_name].values
    lon_min = np.nanmin(lon_values)
    lon_max = np.nanmax(lon_values)

    county_lons = county_meta["rep_lon"].to_numpy()

    # CMIP6 longitudes are often 0..360. Convert county lon if needed.
    if lon_min >= 0 and lon_max > 180:
        county_lons_for_model = (county_lons + 360) % 360
    else:
        county_lons_for_model = county_lons

    county_lats_da = xr.DataArray(
        county_meta["rep_lat"].to_numpy(),
        dims="county_index",
        coords={"county_index": county_meta["GEOID"].to_numpy()},
    )

    county_lons_da = xr.DataArray(
        county_lons_for_model,
        dims="county_index",
        coords={"county_index": county_meta["GEOID"].to_numpy()},
    )

    sampled = da.sel(
        {
            lat_name: county_lats_da,
            lon_name: county_lons_da,
        },
        method="nearest",
    )

    sampled = sampled.assign_coords(county_index=county_meta["GEOID"].to_numpy())
    return sampled


def annual_da_to_county_df(annual_da, value_name, scenario):
    # Convert annual xarray DataArray with dims time and county_index into a clean DataFrame.
    df = annual_da.to_dataframe(name=value_name).reset_index()
    keep_cols = [c for c in ["county_index", "time", value_name] if c in df.columns]
    df = df[keep_cols].copy()

    df["year"] = df["time"].apply(lambda t: t.year)
    df = df[df["year"].isin(ALL_YEARS)].copy()
    df = df.rename(columns={"county_index": "GEOID"})
    df["GEOID"] = df["GEOID"].astype(str).str.zfill(5)
    df["scenario"] = scenario
    df["scenario_label"] = SCENARIO_LABELS[scenario]

    return df[["GEOID", "year", "scenario", "scenario_label", value_name]]


def monthly_da_to_county_df(monthly_da, value_name, scenario):
    # Convert monthly xarray DataArray with dims time and county_index into a clean DataFrame.
    df = monthly_da.to_dataframe(name=value_name).reset_index()
    keep_cols = [c for c in ["county_index", "time", value_name] if c in df.columns]
    df = df[keep_cols].copy()

    df["year"] = df["time"].apply(lambda t: t.year)
    df["month"] = df["time"].apply(lambda t: t.month)
    df = df[df["year"].isin(ALL_YEARS)].copy()
    df = df.rename(columns={"county_index": "GEOID"})
    df["GEOID"] = df["GEOID"].astype(str).str.zfill(5)
    df["scenario"] = scenario
    df["scenario_label"] = SCENARIO_LABELS[scenario]

    return df[["GEOID", "year", "month", "scenario", "scenario_label", value_name]]


def weighted_average(group, value_col, weight_col="ALAND"):
    valid = (
        group[value_col].notna() &
        group[weight_col].notna() &
        (group[weight_col] > 0)
    )
    if valid.sum() == 0:
        return np.nan
    return np.average(group.loc[valid, value_col], weights=group.loc[valid, weight_col])


def aggregate_county_to_state(county_df, value_cols, extra_group_cols=None):
    # Area-weight county rows to state rows using county ALAND.
    if extra_group_cols is None:
        extra_group_cols = ["year"]

    group_cols = ["state", "state_abbr"] + extra_group_cols + ["scenario", "scenario_label"]
    state_records = []

    for keys, group in county_df.groupby(group_cols, dropna=False):
        record = dict(zip(group_cols, keys))
        for col in value_cols:
            record[col] = weighted_average(group, col)
        state_records.append(record)

    sort_cols = ["state", "scenario"] + extra_group_cols
    return pd.DataFrame(state_records).sort_values(sort_cols)


def add_annual_baseline_change(df, id_cols, value_col, baseline_year=BASELINE_YEAR):
    # Add baseline and change-from-baseline columns for annual/summer files.
    baseline = df[df["year"] == baseline_year][id_cols + [value_col]].copy()
    baseline = baseline.rename(columns={value_col: f"{value_col}_2020"})
    out = df.merge(baseline, on=id_cols, how="left")
    out[f"{value_col}_change_from_2020"] = out[value_col] - out[f"{value_col}_2020"]
    return out


def add_monthly_baseline_change(df, id_cols, value_col, baseline_year=BASELINE_YEAR):
    # Add baseline and change-from-baseline columns for monthly files.
    # Baseline is matched by the same calendar month in 2020.
    baseline_cols = id_cols + ["month"]
    baseline = df[df["year"] == baseline_year][baseline_cols + [value_col]].copy()
    baseline = baseline.rename(columns={value_col: f"{value_col}_2020_same_month"})
    out = df.merge(baseline, on=baseline_cols, how="left")
    out[f"{value_col}_change_from_2020_same_month"] = (
        out[value_col] - out[f"{value_col}_2020_same_month"]
    )
    return out

In [8]:

# ============================================================
# Observed temperature, 2000-2020
# Source family: NOAA NCEI nClimGrid (Azure public mirror paths)
#
# Important: this keeps the same logic as the original CMIP6 workflow:
#   - Average temperature uses an average-temperature variable:
#       observed nClimGrid monthly tavg  ↔  CMIP6 monthly tas
#   - Extreme hot days use a daily maximum-temperature variable:
#       observed nClimGrid-Daily tmax    ↔  CMIP6 daily tasmax
#
# We do NOT compute observed average temperature as (tmax + tmin) / 2.
# ============================================================

OBS_DIR = OUTPUT_DIR / "observed_nclimgrid"
OBS_DIR.mkdir(parents=True, exist_ok=True)

# nClimGrid gridded products are CONUS, so use lower-48 counties for observed processing.
county_meta_obs = county_meta[county_meta["state_abbr"].isin(CONUS_STATE_ABBRS)].copy()
county_attrs_obs = county_attrs[county_attrs["state_abbr"].isin(CONUS_STATE_ABBRS)].copy()

NCLIMGRID_BASE_URL = "https://nclimgrideastus.blob.core.windows.net/nclimgrid"
NCLIMGRID_DAILY_BASE_URL = "https://www.ncei.noaa.gov/thredds/fileServer/nclimgrid-daily"

# Observed nClimGrid access paths:
# Monthly tavg uses the public Azure mirror:
#   https://nclimgrideastus.blob.core.windows.net/nclimgrid/nclimgrid-monthly/nclimgrid_tavg.nc
# Daily files use NOAA NCEI THREDDS:
#   https://www.ncei.noaa.gov/thredds/fileServer/nclimgrid-daily/YYYY/ncdd-YYYYMM-grd-scaled.nc
# Each daily monthly file contains variables such as tmax, tmin, tavg, and prcp.

def download_file(url, local_path, overwrite=False):
    """Download a file to OBS_DIR if it does not already exist."""
    local_path = Path(local_path)
    if local_path.exists() and not overwrite:
        print(f"Already downloaded: {local_path.name}")
        return local_path

    print(f"Downloading:\n  {url}\n  -> {local_path}")
    urllib.request.urlretrieve(url, local_path)
    return local_path


def nclimgrid_monthly_tavg_url():
    return f"{NCLIMGRID_BASE_URL}/nclimgrid-monthly/nclimgrid_tavg.nc"


def nclimgrid_daily_tmax_url(year, month):
    """
    Return the NOAA THREDDS download URL for one nClimGrid-Daily monthly file.

    Important: the file is named ncdd-YYYYMM-grd-scaled.nc.
    The daily maximum temperature is the internal variable named tmax.
    """
    return f"{NCLIMGRID_DAILY_BASE_URL}/{year}/ncdd-{year}{month:02d}-grd-scaled.nc"


def get_first_existing_var(ds, candidates):
    """Return the first matching data variable name in an xarray Dataset."""
    for name in candidates:
        if name in ds.data_vars:
            return name
    raise KeyError(
        f"None of these variables were found: {candidates}. "
        f"Available variables: {list(ds.data_vars)}"
    )


def observed_to_county_annual_df(annual_da, value_name):
    df = annual_da.to_dataframe(name=value_name).reset_index()
    keep_cols = [c for c in ["county_index", "time", value_name] if c in df.columns]
    df = df[keep_cols].copy()

    df["year"] = df["time"].apply(lambda t: t.year)
    df = df[(df["year"] >= OBS_START_YEAR) & (df["year"] <= OBS_END_YEAR)].copy()

    df = df.rename(columns={"county_index": "GEOID"})
    df["GEOID"] = df["GEOID"].astype(str).str.zfill(5)
    df["scenario"] = "observed"
    df["scenario_label"] = SCENARIO_LABELS["observed"]
    df["data_source"] = "observed"

    return df[["GEOID", "year", "scenario", "scenario_label", "data_source", value_name]]


def observed_to_county_monthly_df(monthly_da, value_name):
    df = monthly_da.to_dataframe(name=value_name).reset_index()
    keep_cols = [c for c in ["county_index", "time", value_name] if c in df.columns]
    df = df[keep_cols].copy()

    df["year"] = df["time"].apply(lambda t: t.year)
    df["month"] = df["time"].apply(lambda t: t.month)
    df = df[(df["year"] >= OBS_START_YEAR) & (df["year"] <= OBS_END_YEAR)].copy()

    df = df.rename(columns={"county_index": "GEOID"})
    df["GEOID"] = df["GEOID"].astype(str).str.zfill(5)
    df["scenario"] = "observed"
    df["scenario_label"] = SCENARIO_LABELS["observed"]
    df["data_source"] = "observed"

    return df[["GEOID", "year", "month", "scenario", "scenario_label", "data_source", value_name]]


def standardize_temperature_to_celsius(da):
    units = str(da.attrs.get("units", "")).lower()

    if "fahrenheit" in units or units.strip() in {"f", "degf", "degree_fahrenheit"}:
        return (da - 32) * 5 / 9

    if units.strip() in {"k", "kelvin"}:
        return da - 273.15

    # nClimGrid temperature variables are commonly stored in °C.
    return da


def open_nclimgrid_monthly_tavg():
    """
    Open NOAA nClimGrid monthly average temperature.

    This is the observed average-temperature product used for the average-warming part
    of the story. It is not derived from daily Tmax/Tmin in this notebook.
    """
    url = nclimgrid_monthly_tavg_url()
    local_path = OBS_DIR / "nclimgrid_tavg.nc"
    download_file(url, local_path)

    ds = xr.open_dataset(local_path, chunks={"time": 12})
    if "time" in ds.coords:
        ds = ds.sel(time=slice(f"{OBS_START_YEAR}-01-01", f"{OBS_END_YEAR}-12-31"))
    return ds


def open_nclimgrid_daily_tmax_for_year(year):
    """
    Open NOAA nClimGrid-Daily for one year by downloading the 12 monthly files.

    File names are ncdd-YYYYMM-grd-scaled.nc. We then read the tmax variable
    inside each file for observed daily maximum temperature.
    """
    local_files = []
    for month in range(1, 13):
        url = nclimgrid_daily_tmax_url(year, month)
        local_path = OBS_DIR / f"ncdd-{year}{month:02d}-grd-scaled.nc"
        download_file(url, local_path)
        local_files.append(local_path)

    ds = xr.open_mfdataset(
        local_files,
        combine="by_coords",
        chunks={"time": 31},
    )
    if "time" in ds.coords:
        ds = ds.sel(time=slice(f"{year}-01-01", f"{year}-12-31"))
    return ds


print("\n==============================")
print("Processing observed nClimGrid average temperature and 35°C+ days, 2000-2020")
print("==============================")

# -----------------------
# Observed average temperature from nClimGrid monthly tavg.
# This mirrors the original CMIP6 average-temperature workflow:
#   CMIP6: monthly tas
#   Observed: monthly tavg
# -----------------------
obs_tavg_ds = open_nclimgrid_monthly_tavg()
obs_tavg_var = get_first_existing_var(obs_tavg_ds, ["tavg", "TAVG"])
print("Observed monthly average temperature variable:", obs_tavg_var)

obs_lat_name = _get_coord_name(obs_tavg_ds, ["lat", "latitude", "y"])
obs_lon_name = _get_coord_name(obs_tavg_ds, ["lon", "longitude", "x"])

obs_tavg = obs_tavg_ds[obs_tavg_var].sel(
    time=slice(f"{OBS_START_YEAR}-01-01", f"{OBS_END_YEAR}-12-31")
)
obs_tavg_c = standardize_temperature_to_celsius(obs_tavg)
obs_tavg_c.name = "observed_tavg_c"

sampled_obs_tavg_c = sample_county_timeseries(
    obs_tavg_c,
    obs_lat_name,
    obs_lon_name,
    county_meta_obs,
)

# Annual observed average temperature.
observed_annual_tas_c = sampled_obs_tavg_c.resample(time="YS").mean()
observed_annual_tas_county = observed_to_county_annual_df(
    observed_annual_tas_c,
    "tas_c",
)

# Summer observed average temperature.
observed_summer_tas_c = (
    sampled_obs_tavg_c
    .where(sampled_obs_tavg_c["time"].dt.month.isin(SUMMER_MONTHS), drop=True)
    .resample(time="YS")
    .mean()
)
observed_summer_tas_county = observed_to_county_annual_df(
    observed_summer_tas_c,
    "summer_tas_c",
)
observed_summer_tas_county["season"] = SUMMER_LABEL

# Monthly observed average temperature.
observed_monthly_tas_c = sampled_obs_tavg_c.resample(time="MS").mean()
observed_monthly_tas_county = observed_to_county_monthly_df(
    observed_monthly_tas_c,
    "monthly_tas_c",
)

# Aggregate observed average temperature to states.
observed_annual_tas_county = observed_annual_tas_county.merge(county_attrs_obs, on="GEOID", how="left")
observed_summer_tas_county = observed_summer_tas_county.merge(county_attrs_obs, on="GEOID", how="left")
observed_monthly_tas_county = observed_monthly_tas_county.merge(county_attrs_obs, on="GEOID", how="left")

state_observed_annual_tas = aggregate_county_to_state(
    observed_annual_tas_county,
    value_cols=["tas_c"],
    extra_group_cols=["year", "data_source"],
)

state_observed_summer_tas = aggregate_county_to_state(
    observed_summer_tas_county,
    value_cols=["summer_tas_c"],
    extra_group_cols=["year", "season", "data_source"],
)

state_observed_monthly_tas = aggregate_county_to_state(
    observed_monthly_tas_county,
    value_cols=["monthly_tas_c"],
    extra_group_cols=["year", "month", "data_source"],
)

# -----------------------
# Observed extreme hot days from nClimGrid-Daily tmax.
# This mirrors the original CMIP6 extreme-heat workflow:
#   CMIP6: daily tasmax
#   Observed: daily tmax
# -----------------------
observed_annual_heat_parts = []
observed_summer_heat_parts = []
observed_monthly_heat_parts = []

for year in range(OBS_START_YEAR, OBS_END_YEAR + 1):
    print(f"Processing observed daily tmax for {year}")

    obs_tmax_ds = open_nclimgrid_daily_tmax_for_year(year)
    obs_tmax_var = get_first_existing_var(obs_tmax_ds, ["tmax", "TMAX"])
    print("Observed daily maximum temperature variable:", obs_tmax_var)

    obs_tmax_lat_name = _get_coord_name(obs_tmax_ds, ["lat", "latitude", "y"])
    obs_tmax_lon_name = _get_coord_name(obs_tmax_ds, ["lon", "longitude", "x"])

    obs_tmax = obs_tmax_ds[obs_tmax_var].sel(
        time=slice(f"{year}-01-01", f"{year}-12-31")
    )
    obs_tmax_c = standardize_temperature_to_celsius(obs_tmax)
    obs_tmax_c.name = "observed_tmax_c"

    sampled_obs_tmax_c = sample_county_timeseries(
        obs_tmax_c,
        obs_tmax_lat_name,
        obs_tmax_lon_name,
        county_meta_obs,
    )

    # Annual max of daily maximum temperature.
    observed_annual_max = sampled_obs_tmax_c.resample(time="YS").max()
    observed_annual_max_df = observed_to_county_annual_df(
        observed_annual_max,
        "annual_max_tasmax_c",
    )

    observed_annual_combined = observed_annual_max_df.copy()

    for threshold in HOT_THRESHOLDS_C:
        value_name = f"hot_days_{threshold}c"
        observed_hot_days = (sampled_obs_tmax_c > threshold).resample(time="YS").sum()
        observed_hot_df = observed_to_county_annual_df(
            observed_hot_days,
            value_name,
        )
        observed_annual_combined = observed_annual_combined.merge(
            observed_hot_df,
            on=["GEOID", "year", "scenario", "scenario_label", "data_source"],
            how="left",
        )

    observed_annual_heat_parts.append(observed_annual_combined)

    # Summer-only max and hot-day counts.
    sampled_obs_summer_tmax_c = sampled_obs_tmax_c.where(
        sampled_obs_tmax_c["time"].dt.month.isin(SUMMER_MONTHS),
        drop=True,
    )

    observed_summer_max = sampled_obs_summer_tmax_c.resample(time="YS").max()
    observed_summer_max_df = observed_to_county_annual_df(
        observed_summer_max,
        "summer_max_tasmax_c",
    )
    observed_summer_max_df["season"] = SUMMER_LABEL

    observed_summer_combined = observed_summer_max_df.copy()

    for threshold in HOT_THRESHOLDS_C:
        value_name = f"summer_hot_days_{threshold}c"
        observed_summer_hot_days = (sampled_obs_summer_tmax_c > threshold).resample(time="YS").sum()
        observed_summer_hot_df = observed_to_county_annual_df(
            observed_summer_hot_days,
            value_name,
        )
        observed_summer_hot_df["season"] = SUMMER_LABEL
        observed_summer_combined = observed_summer_combined.merge(
            observed_summer_hot_df,
            on=["GEOID", "year", "season", "scenario", "scenario_label", "data_source"],
            how="left",
        )

    observed_summer_heat_parts.append(observed_summer_combined)

    # Monthly max and hot-day counts.
    observed_monthly_max = sampled_obs_tmax_c.resample(time="MS").max()
    observed_monthly_combined = observed_to_county_monthly_df(
        observed_monthly_max,
        "monthly_max_tasmax_c",
    )

    for threshold in HOT_THRESHOLDS_C:
        value_name = f"monthly_hot_days_{threshold}c"
        observed_monthly_hot_days = (sampled_obs_tmax_c > threshold).resample(time="MS").sum()
        observed_monthly_hot_df = observed_to_county_monthly_df(
            observed_monthly_hot_days,
            value_name,
        )
        observed_monthly_combined = observed_monthly_combined.merge(
            observed_monthly_hot_df,
            on=["GEOID", "year", "month", "scenario", "scenario_label", "data_source"],
            how="left",
        )

    observed_monthly_heat_parts.append(observed_monthly_combined)

    del obs_tmax_ds, obs_tmax, obs_tmax_c, sampled_obs_tmax_c
    gc.collect()

observed_annual_heat_county = pd.concat(observed_annual_heat_parts, ignore_index=True)
observed_summer_heat_county = pd.concat(observed_summer_heat_parts, ignore_index=True)
observed_monthly_heat_county = pd.concat(observed_monthly_heat_parts, ignore_index=True)

observed_annual_heat_county = observed_annual_heat_county.merge(county_attrs_obs, on="GEOID", how="left")
observed_summer_heat_county = observed_summer_heat_county.merge(county_attrs_obs, on="GEOID", how="left")
observed_monthly_heat_county = observed_monthly_heat_county.merge(county_attrs_obs, on="GEOID", how="left")

state_observed_annual_heat = aggregate_county_to_state(
    observed_annual_heat_county,
    value_cols=["annual_max_tasmax_c", "hot_days_30c", "hot_days_35c"],
    extra_group_cols=["year", "data_source"],
)

state_observed_summer_heat = aggregate_county_to_state(
    observed_summer_heat_county,
    value_cols=["summer_max_tasmax_c", "summer_hot_days_30c", "summer_hot_days_35c"],
    extra_group_cols=["year", "season", "data_source"],
)

state_observed_monthly_heat = aggregate_county_to_state(
    observed_monthly_heat_county,
    value_cols=["monthly_max_tasmax_c", "monthly_hot_days_30c", "monthly_hot_days_35c"],
    extra_group_cols=["year", "month", "data_source"],
)

# -----------------------
# Combine observed average temperature and observed hot-day metrics.
# -----------------------
state_observed_annual_combined = state_observed_annual_tas.merge(
    state_observed_annual_heat,
    on=["state", "state_abbr", "year", "scenario", "scenario_label", "data_source"],
    how="left",
)

state_observed_summer_combined = state_observed_summer_tas.merge(
    state_observed_summer_heat,
    on=["state", "state_abbr", "year", "season", "scenario", "scenario_label", "data_source"],
    how="left",
)

state_observed_monthly_combined = state_observed_monthly_tas.merge(
    state_observed_monthly_heat,
    on=["state", "state_abbr", "year", "month", "scenario", "scenario_label", "data_source"],
    how="left",
)

print("Observed annual combined:")
display(state_observed_annual_combined.head())

print("Observed summer combined:")
display(state_observed_summer_combined.head())

print("Observed monthly combined:")
display(state_observed_monthly_combined.head())



Processing observed nClimGrid average temperature and 35°C+ days, 2000-2020
Downloading:
  https://nclimgrideastus.blob.core.windows.net/nclimgrid/nclimgrid-monthly/nclimgrid_tavg.nc
  -> /content/cmip6_heat_story_outputs/observed_nclimgrid/nclimgrid_tavg.nc
Observed monthly average temperature variable: tavg
Processing observed daily tmax for 2000
Downloading:
  https://www.ncei.noaa.gov/thredds/fileServer/nclimgrid-daily/2000/ncdd-200001-grd-scaled.nc
  -> /content/cmip6_heat_story_outputs/observed_nclimgrid/ncdd-200001-grd-scaled.nc
Downloading:
  https://www.ncei.noaa.gov/thredds/fileServer/nclimgrid-daily/2000/ncdd-200002-grd-scaled.nc
  -> /content/cmip6_heat_story_outputs/observed_nclimgrid/ncdd-200002-grd-scaled.nc
Downloading:
  https://www.ncei.noaa.gov/thredds/fileServer/nclimgrid-daily/2000/ncdd-200003-grd-scaled.nc
  -> /content/cmip6_heat_story_outputs/observed_nclimgrid/ncdd-200003-grd-scaled.nc
Downloading:
  https://www.ncei.noaa.gov/thredds/fileServer/nclimgrid-daily

,state,state_abbr,year,data_source,scenario,scenario_label,tas_c,annual_max_tasmax_c,hot_days_30c,hot_days_35c
0,Alabama,AL,2000,observed,observed,Observed,17.555523,39.077120,120.930642,32.087657
1,Alabama,AL,2001,observed,observed,Observed,17.254575,34.334240,90.703641,0.076583
2,Alabama,AL,2002,observed,observed,Observed,17.561072,35.958477,122.218428,6.467887
3,Alabama,AL,2003,observed,observed,Observed,17.099079,33.810311,85.772614,0.000000
4,Alabama,AL,2004,observed,observed,Observed,17.421374,35.047462,86.944867,1.033682


Observed summer combined:


,state,state_abbr,year,season,data_source,scenario,scenario_label,summer_tas_c,summer_max_tasmax_c,summer_hot_days_30c,summer_hot_days_35c
0,Alabama,AL,2000,JJA,observed,observed,Observed,26.768856,39.077120,84.880751,30.481202
1,Alabama,AL,2001,JJA,observed,observed,Observed,25.558120,34.334240,69.370460,0.076583
2,Alabama,AL,2002,JJA,observed,observed,Observed,26.156654,35.683935,82.693427,4.162342
3,Alabama,AL,2003,JJA,observed,observed,Observed,25.491628,33.810311,66.422927,0.000000
4,Alabama,AL,2004,JJA,observed,observed,Observed,25.262000,35.047462,61.200298,1.033682


Observed monthly combined:


,state,state_abbr,year,month,data_source,scenario,scenario_label,monthly_tas_c,monthly_max_tasmax_c,monthly_hot_days_30c,monthly_hot_days_35c
0,Alabama,AL,2000,1,observed,observed,Observed,8.089767,23.095223,0.000000,0.000000
1,Alabama,AL,2000,2,observed,observed,Observed,11.288313,25.764344,0.000000,0.000000
2,Alabama,AL,2000,3,observed,observed,Observed,14.959017,27.245241,0.000000,0.000000
3,Alabama,AL,2000,4,observed,observed,Observed,15.637973,28.730756,0.051440,0.000000
4,Alabama,AL,2000,5,observed,observed,Observed,23.329185,33.590543,16.335073,0.212295


In [9]:

def add_observed_and_cmip6_2020_deltas(
    df,
    observed_baseline_df,
    value_col,
    id_cols,
    scenario_col="scenario",
    data_source_col="data_source",
    baseline_year=BASELINE_YEAR,
    monthly=False,
):
    """
    Add observed 2020 baseline, CMIP6 2020 baseline, and delta columns.

    Creates:
      {value_col}_observed_2020
      {value_col}_cmip6_2020
      {value_col}_change_from_observed_2020
      {value_col}_change_from_cmip6_2020
      {value_col}_cmip6_2020_minus_observed_2020

    For annual/summer:
      observed baseline is matched by state.
      CMIP6 baseline is matched by state + scenario.

    For monthly:
      observed baseline is matched by state + month.
      CMIP6 baseline is matched by state + month + scenario.
    """

    baseline_cols = id_cols.copy()
    if monthly:
        baseline_cols = baseline_cols + ["month"]

    # Observed 2020 baseline.
    observed_baseline = observed_baseline_df[
        observed_baseline_df["year"] == baseline_year
    ][baseline_cols + [value_col]].copy()

    observed_baseline = observed_baseline.rename(
        columns={value_col: f"{value_col}_observed_2020"}
    )

    out = df.merge(
        observed_baseline,
        on=baseline_cols,
        how="left",
    )

    # CMIP6 2020 baseline, matched within each scenario.
    cmip6_baseline_cols = baseline_cols + [scenario_col]

    cmip6_baseline = df[
        (df["year"] == baseline_year)
        & (df[data_source_col] == "projected")
    ][cmip6_baseline_cols + [value_col]].copy()

    cmip6_baseline = cmip6_baseline.rename(
        columns={value_col: f"{value_col}_cmip6_2020"}
    )

    out = out.merge(
        cmip6_baseline,
        on=cmip6_baseline_cols,
        how="left",
    )

    out[f"{value_col}_change_from_observed_2020"] = (
        out[value_col] - out[f"{value_col}_observed_2020"]
    )

    out[f"{value_col}_change_from_cmip6_2020"] = (
        out[value_col] - out[f"{value_col}_cmip6_2020"]
    )

    out[f"{value_col}_cmip6_2020_minus_observed_2020"] = (
        out[f"{value_col}_cmip6_2020"] - out[f"{value_col}_observed_2020"]
    )

    return out


## 5. Build average temperature files from `tas`

This section uses `tas` from CMIP6 monthly data (`Amon`).

It creates:

- annual average temperature
- summer-only average temperature
- monthly average temperature

In [10]:
county_tas_parts = []
county_tas_summer_parts = []
state_tas_monthly_parts = []

for scenario in SCENARIOS:
    print("\n==============================")
    print("Processing tas:", scenario)
    print("==============================")

    tas, lat_name, lon_name = open_cmip6_variable(
        scenario=scenario,
        variable_id="tas",
        table_id="Amon",
    )

    sampled_tas = sample_county_timeseries(tas, lat_name, lon_name, county_meta)
    sampled_tas_c = to_celsius(sampled_tas)
    sampled_tas_c.name = "tas_c"

    # -----------------------
    # Annual average tas
    # -----------------------
    annual_tas_c = sampled_tas_c.resample(time="YS").mean()
    county_scenario_tas = annual_da_to_county_df(
        annual_da=annual_tas_c,
        value_name="tas_c",
        scenario=scenario,
    )
    county_tas_parts.append(county_scenario_tas)

    # -----------------------
    # Summer average tas
    # -----------------------
    summer_tas_c = (
        sampled_tas_c
        .where(sampled_tas_c["time"].dt.month.isin(SUMMER_MONTHS), drop=True)
        .resample(time="YS")
        .mean()
    )
    county_scenario_summer_tas = annual_da_to_county_df(
        annual_da=summer_tas_c,
        value_name="summer_tas_c",
        scenario=scenario,
    )
    county_scenario_summer_tas["season"] = SUMMER_LABEL
    county_tas_summer_parts.append(county_scenario_summer_tas)

    # -----------------------
    # Monthly average tas
    # -----------------------
    # tas is already monthly, but resample to MS makes the timestamp consistent.
    monthly_tas_c = sampled_tas_c.resample(time="MS").mean()
    county_scenario_monthly_tas = monthly_da_to_county_df(
        monthly_da=monthly_tas_c,
        value_name="monthly_tas_c",
        scenario=scenario,
    )
    county_scenario_monthly_tas = county_scenario_monthly_tas.merge(county_attrs, on="GEOID", how="left")
    state_scenario_monthly_tas = aggregate_county_to_state(
        county_scenario_monthly_tas,
        value_cols=["monthly_tas_c"],
        extra_group_cols=["year", "month"],
    )
    state_tas_monthly_parts.append(state_scenario_monthly_tas)

    # Free memory before next scenario.
    del tas, sampled_tas, sampled_tas_c
    del annual_tas_c, county_scenario_tas
    del summer_tas_c, county_scenario_summer_tas
    del monthly_tas_c, county_scenario_monthly_tas, state_scenario_monthly_tas
    gc.collect()

# -----------------------
# Annual tas: county + state
# -----------------------
county_tas = pd.concat(county_tas_parts, ignore_index=True)
county_tas = county_tas.merge(county_attrs, on="GEOID", how="left")

county_tas = add_annual_baseline_change(
    county_tas,
    id_cols=["GEOID", "scenario"],
    value_col="tas_c",
)

county_tas = county_tas[
    [
        "GEOID", "county", "state", "state_abbr", "ALAND",
        "year", "scenario", "scenario_label",
        "tas_c", "tas_c_2020", "tas_c_change_from_2020",
    ]
].rename(columns={
    "tas_c_2020": "tas_2020_c",
    "tas_c_change_from_2020": "tas_change_from_2020_c",
}).sort_values(["state", "county", "scenario", "year"])

state_tas = aggregate_county_to_state(
    county_tas,
    value_cols=["tas_c", "tas_2020_c", "tas_change_from_2020_c"],
    extra_group_cols=["year"],
)

# -----------------------
# Summer tas: county + state
# -----------------------
county_summer_tas = pd.concat(county_tas_summer_parts, ignore_index=True)
county_summer_tas = county_summer_tas.merge(county_attrs, on="GEOID", how="left")

county_summer_tas = add_annual_baseline_change(
    county_summer_tas,
    id_cols=["GEOID", "scenario"],
    value_col="summer_tas_c",
)

county_summer_tas = county_summer_tas[
    [
        "GEOID", "county", "state", "state_abbr", "ALAND",
        "year", "season", "scenario", "scenario_label",
        "summer_tas_c", "summer_tas_c_2020", "summer_tas_c_change_from_2020",
    ]
].sort_values(["state", "county", "scenario", "year"])

state_summer_tas = aggregate_county_to_state(
    county_summer_tas,
    value_cols=["summer_tas_c", "summer_tas_c_2020", "summer_tas_c_change_from_2020"],
    extra_group_cols=["year", "season"],
)

# -----------------------
# Monthly tas: state only
# -----------------------
state_monthly_tas = pd.concat(state_tas_monthly_parts, ignore_index=True)
state_monthly_tas = add_monthly_baseline_change(
    state_monthly_tas,
    id_cols=["state", "state_abbr", "scenario"],
    value_col="monthly_tas_c",
)

print("county_tas rows:", len(county_tas))
print("state_tas rows:", len(state_tas))
print("state_summer_tas rows:", len(state_summer_tas))
print("state_monthly_tas rows:", len(state_monthly_tas))

display(state_tas.head())
display(state_summer_tas.head())
display(state_monthly_tas.head())


Processing tas: ssp126
Found 1 catalog entries for ssp126, tas, Amon. Using first matching dataset.

--> The keys in the returned dictionary of datasets are constructed as follows:
	'activity_id.institution_id.source_id.experiment_id.table_id.grid_label'


<div><progress max="1" value="1"></progress> 100.00% [1/1 00:04&lt;00:00]</div>


Processing tas: ssp245
Found 1 catalog entries for ssp245, tas, Amon. Using first matching dataset.

--> The keys in the returned dictionary of datasets are constructed as follows:
	'activity_id.institution_id.source_id.experiment_id.table_id.grid_label'


<div><progress max="1" value="1"></progress> 100.00% [1/1 00:00&lt;00:00]</div>


Processing tas: ssp585
Found 1 catalog entries for ssp585, tas, Amon. Using first matching dataset.

--> The keys in the returned dictionary of datasets are constructed as follows:
	'activity_id.institution_id.source_id.experiment_id.table_id.grid_label'


<div><progress max="1" value="1"></progress> 100.00% [1/1 00:00&lt;00:00]</div>

county_tas rows: 763749
state_tas rows: 12150
state_summer_tas rows: 12150
state_monthly_tas rows: 145800


,state,state_abbr,year,scenario,scenario_label,tas_c,tas_2020_c,tas_change_from_2020_c
0,Alabama,AL,2020,ssp126,Low emissions (SSP126),15.809565,15.809565,0.000000
3,Alabama,AL,2021,ssp126,Low emissions (SSP126),16.903114,15.809565,1.093548
6,Alabama,AL,2022,ssp126,Low emissions (SSP126),16.508223,15.809565,0.698658
9,Alabama,AL,2023,ssp126,Low emissions (SSP126),16.367164,15.809565,0.557599
12,Alabama,AL,2024,ssp126,Low emissions (SSP126),17.459909,15.809565,1.650343


,state,state_abbr,year,season,scenario,scenario_label,summer_tas_c,summer_tas_c_2020,summer_tas_c_change_from_2020
0,Alabama,AL,2020,JJA,ssp126,Low emissions (SSP126),24.680512,24.680512,0.000000
3,Alabama,AL,2021,JJA,ssp126,Low emissions (SSP126),28.238231,24.680512,3.557719
6,Alabama,AL,2022,JJA,ssp126,Low emissions (SSP126),25.321384,24.680512,0.640872
9,Alabama,AL,2023,JJA,ssp126,Low emissions (SSP126),25.333490,24.680512,0.652979
12,Alabama,AL,2024,JJA,ssp126,Low emissions (SSP126),26.772844,24.680512,2.092333


,state,state_abbr,year,month,scenario,scenario_label,monthly_tas_c,monthly_tas_c_2020_same_month,monthly_tas_c_change_from_2020_same_month
0,Alabama,AL,2020,1,ssp126,Low emissions (SSP126),7.261339,7.261339,0.0
1,Alabama,AL,2020,2,ssp126,Low emissions (SSP126),9.080862,9.080862,0.0
2,Alabama,AL,2020,3,ssp126,Low emissions (SSP126),11.956863,11.956863,0.0
3,Alabama,AL,2020,4,ssp126,Low emissions (SSP126),18.088153,18.088153,0.0
4,Alabama,AL,2020,5,ssp126,Low emissions (SSP126),19.890474,19.890474,0.0


In [11]:
county_tas_csv = OUTPUT_DIR / "county_annual_tas_cmip6_2020_2070.csv"
state_tas_csv = OUTPUT_DIR / "state_annual_tas_cmip6_2020_2070.csv"
state_summer_tas_csv = OUTPUT_DIR / "state_summer_tas_cmip6_2020_2070.csv"
state_monthly_tas_csv = OUTPUT_DIR / "state_monthly_tas_cmip6_2020_2070.csv"

county_tas.to_csv(county_tas_csv, index=False)
state_tas.to_csv(state_tas_csv, index=False)
state_summer_tas.to_csv(state_summer_tas_csv, index=False)
state_monthly_tas.to_csv(state_monthly_tas_csv, index=False)

print("Saved:")
print(county_tas_csv)
print(state_tas_csv)
print(state_summer_tas_csv)
print(state_monthly_tas_csv)

Saved:
/content/cmip6_heat_story_outputs/county_annual_tas_cmip6_2020_2070.csv
/content/cmip6_heat_story_outputs/state_annual_tas_cmip6_2020_2070.csv
/content/cmip6_heat_story_outputs/state_summer_tas_cmip6_2020_2070.csv
/content/cmip6_heat_story_outputs/state_monthly_tas_cmip6_2020_2070.csv


## 6. Build extreme hot day files from daily `tasmax`

This section uses `tasmax` from CMIP6 daily data (`day`).

It creates:

- annual maximum daily max temperature
- annual number of days above 30°C and 35°C
- summer-only number of days above 30°C and 35°C
- monthly number of days above 30°C and 35°C

This cell can take longer because `tasmax` is daily data.

In [12]:
county_heat_parts = []
county_summer_heat_parts = []
state_monthly_heat_parts = []

for scenario in SCENARIOS:
    print("\n==============================")
    print("Processing daily tasmax:", scenario)
    print("==============================")

    tasmax, lat_name, lon_name = open_cmip6_variable(
        scenario=scenario,
        variable_id="tasmax",
        table_id="day",
    )

    sampled_tasmax = sample_county_timeseries(tasmax, lat_name, lon_name, county_meta)
    sampled_tasmax_c = to_celsius(sampled_tasmax)
    sampled_tasmax_c.name = "tasmax_c"

    # -----------------------
    # Annual max of daily maximum temperature
    # -----------------------
    annual_max_tasmax_c = sampled_tasmax_c.resample(time="YS").max()
    annual_max_df = annual_da_to_county_df(
        annual_da=annual_max_tasmax_c,
        value_name="annual_max_tasmax_c",
        scenario=scenario,
    )

    # -----------------------
    # Annual hot day counts
    # -----------------------
    annual_hot_count_dfs = []
    for threshold in HOT_THRESHOLDS_C:
        value_name = f"hot_days_{threshold}c"
        print(f"  Annual count: days above {threshold}°C")
        hot_days = (sampled_tasmax_c > threshold).resample(time="YS").sum()
        hot_df = annual_da_to_county_df(
            annual_da=hot_days,
            value_name=value_name,
            scenario=scenario,
        )
        annual_hot_count_dfs.append(hot_df)

    annual_combined = annual_max_df.copy()
    for hot_df in annual_hot_count_dfs:
        annual_combined = annual_combined.merge(
            hot_df,
            on=["GEOID", "year", "scenario", "scenario_label"],
            how="left",
        )
    county_heat_parts.append(annual_combined)

    # -----------------------
    # Summer-only max and hot day counts
    # -----------------------
    summer_tasmax_c = sampled_tasmax_c.where(
        sampled_tasmax_c["time"].dt.month.isin(SUMMER_MONTHS),
        drop=True,
    )

    summer_max_tasmax_c = summer_tasmax_c.resample(time="YS").max()
    summer_max_df = annual_da_to_county_df(
        annual_da=summer_max_tasmax_c,
        value_name="summer_max_tasmax_c",
        scenario=scenario,
    )
    summer_max_df["season"] = SUMMER_LABEL

    summer_hot_count_dfs = []
    for threshold in HOT_THRESHOLDS_C:
        value_name = f"summer_hot_days_{threshold}c"
        print(f"  Summer count: days above {threshold}°C")
        summer_hot_days = (summer_tasmax_c > threshold).resample(time="YS").sum()
        summer_hot_df = annual_da_to_county_df(
            annual_da=summer_hot_days,
            value_name=value_name,
            scenario=scenario,
        )
        summer_hot_df["season"] = SUMMER_LABEL
        summer_hot_count_dfs.append(summer_hot_df)

    summer_combined = summer_max_df.copy()
    for hot_df in summer_hot_count_dfs:
        summer_combined = summer_combined.merge(
            hot_df,
            on=["GEOID", "year", "season", "scenario", "scenario_label"],
            how="left",
        )
    county_summer_heat_parts.append(summer_combined)

    # -----------------------
    # Monthly max and hot day counts, state only
    # -----------------------
    monthly_max_tasmax_c = sampled_tasmax_c.resample(time="MS").max()
    county_monthly_heat = monthly_da_to_county_df(
        monthly_da=monthly_max_tasmax_c,
        value_name="monthly_max_tasmax_c",
        scenario=scenario,
    )

    for threshold in HOT_THRESHOLDS_C:
        value_name = f"monthly_hot_days_{threshold}c"
        print(f"  Monthly count: days above {threshold}°C")
        monthly_hot_days = (sampled_tasmax_c > threshold).resample(time="MS").sum()
        monthly_hot_df = monthly_da_to_county_df(
            monthly_da=monthly_hot_days,
            value_name=value_name,
            scenario=scenario,
        )
        county_monthly_heat = county_monthly_heat.merge(
            monthly_hot_df,
            on=["GEOID", "year", "month", "scenario", "scenario_label"],
            how="left",
        )

    county_monthly_heat = county_monthly_heat.merge(county_attrs, on="GEOID", how="left")
    state_scenario_monthly_heat = aggregate_county_to_state(
        county_monthly_heat,
        value_cols=["monthly_max_tasmax_c"] + [f"monthly_hot_days_{t}c" for t in HOT_THRESHOLDS_C],
        extra_group_cols=["year", "month"],
    )
    state_monthly_heat_parts.append(state_scenario_monthly_heat)

    # Free memory before next scenario.
    del tasmax, sampled_tasmax, sampled_tasmax_c
    del annual_max_tasmax_c, annual_max_df, annual_hot_count_dfs, annual_combined
    del summer_tasmax_c, summer_max_tasmax_c, summer_max_df, summer_hot_count_dfs, summer_combined
    del monthly_max_tasmax_c, county_monthly_heat, state_scenario_monthly_heat
    gc.collect()

# -----------------------
# Annual heat: county + state
# -----------------------
county_heat = pd.concat(county_heat_parts, ignore_index=True)
county_heat = county_heat.merge(county_attrs, on="GEOID", how="left")

for threshold in HOT_THRESHOLDS_C:
    county_heat = add_annual_baseline_change(
        county_heat,
        id_cols=["GEOID", "scenario"],
        value_col=f"hot_days_{threshold}c",
    )

heat_cols = [
    "GEOID", "county", "state", "state_abbr", "ALAND",
    "year", "scenario", "scenario_label",
    "annual_max_tasmax_c",
]

for threshold in HOT_THRESHOLDS_C:
    heat_cols += [
        f"hot_days_{threshold}c",
        f"hot_days_{threshold}c_2020",
        f"hot_days_{threshold}c_change_from_2020",
    ]

county_heat = county_heat[heat_cols].sort_values(["state", "county", "scenario", "year"])

state_heat_value_cols = ["annual_max_tasmax_c"]
for threshold in HOT_THRESHOLDS_C:
    state_heat_value_cols += [
        f"hot_days_{threshold}c",
        f"hot_days_{threshold}c_2020",
        f"hot_days_{threshold}c_change_from_2020",
    ]

state_heat = aggregate_county_to_state(
    county_heat,
    value_cols=state_heat_value_cols,
    extra_group_cols=["year"],
)

# -----------------------
# Summer heat: county + state
# -----------------------
county_summer_heat = pd.concat(county_summer_heat_parts, ignore_index=True)
county_summer_heat = county_summer_heat.merge(county_attrs, on="GEOID", how="left")

for threshold in HOT_THRESHOLDS_C:
    county_summer_heat = add_annual_baseline_change(
        county_summer_heat,
        id_cols=["GEOID", "scenario"],
        value_col=f"summer_hot_days_{threshold}c",
    )

summer_heat_cols = [
    "GEOID", "county", "state", "state_abbr", "ALAND",
    "year", "season", "scenario", "scenario_label",
    "summer_max_tasmax_c",
]

for threshold in HOT_THRESHOLDS_C:
    summer_heat_cols += [
        f"summer_hot_days_{threshold}c",
        f"summer_hot_days_{threshold}c_2020",
        f"summer_hot_days_{threshold}c_change_from_2020",
    ]

county_summer_heat = county_summer_heat[summer_heat_cols].sort_values(["state", "county", "scenario", "year"])

state_summer_heat_value_cols = ["summer_max_tasmax_c"]
for threshold in HOT_THRESHOLDS_C:
    state_summer_heat_value_cols += [
        f"summer_hot_days_{threshold}c",
        f"summer_hot_days_{threshold}c_2020",
        f"summer_hot_days_{threshold}c_change_from_2020",
    ]

state_summer_heat = aggregate_county_to_state(
    county_summer_heat,
    value_cols=state_summer_heat_value_cols,
    extra_group_cols=["year", "season"],
)

# -----------------------
# Monthly heat: state only
# -----------------------
state_monthly_heat = pd.concat(state_monthly_heat_parts, ignore_index=True)

for threshold in HOT_THRESHOLDS_C:
    state_monthly_heat = add_monthly_baseline_change(
        state_monthly_heat,
        id_cols=["state", "state_abbr", "scenario"],
        value_col=f"monthly_hot_days_{threshold}c",
    )

print("county_heat rows:", len(county_heat))
print("state_heat rows:", len(state_heat))
print("state_summer_heat rows:", len(state_summer_heat))
print("state_monthly_heat rows:", len(state_monthly_heat))

display(state_heat.head())
display(state_summer_heat.head())
display(state_monthly_heat.head())


Processing daily tasmax: ssp126
Found 1 catalog entries for ssp126, tasmax, day. Using first matching dataset.

--> The keys in the returned dictionary of datasets are constructed as follows:
	'activity_id.institution_id.source_id.experiment_id.table_id.grid_label'


<div><progress max="1" value="1"></progress> 100.00% [1/1 00:01&lt;00:00]</div>

  Annual count: days above 30°C
  Annual count: days above 35°C
  Summer count: days above 30°C
  Summer count: days above 35°C
  Monthly count: days above 30°C
  Monthly count: days above 35°C

Processing daily tasmax: ssp245
Found 1 catalog entries for ssp245, tasmax, day. Using first matching dataset.

--> The keys in the returned dictionary of datasets are constructed as follows:
	'activity_id.institution_id.source_id.experiment_id.table_id.grid_label'


<div><progress max="1" value="1"></progress> 100.00% [1/1 00:01&lt;00:00]</div>

  Annual count: days above 30°C
  Annual count: days above 35°C
  Summer count: days above 30°C
  Summer count: days above 35°C
  Monthly count: days above 30°C
  Monthly count: days above 35°C

Processing daily tasmax: ssp585
Found 1 catalog entries for ssp585, tasmax, day. Using first matching dataset.

--> The keys in the returned dictionary of datasets are constructed as follows:
	'activity_id.institution_id.source_id.experiment_id.table_id.grid_label'


<div><progress max="1" value="1"></progress> 100.00% [1/1 00:01&lt;00:00]</div>

  Annual count: days above 30°C
  Annual count: days above 35°C
  Summer count: days above 30°C
  Summer count: days above 35°C
  Monthly count: days above 30°C
  Monthly count: days above 35°C
county_heat rows: 763749
state_heat rows: 12150
state_summer_heat rows: 12150
state_monthly_heat rows: 145800


,state,state_abbr,year,scenario,scenario_label,annual_max_tasmax_c,hot_days_30c,hot_days_30c_2020,hot_days_30c_change_from_2020,hot_days_35c,hot_days_35c_2020,hot_days_35c_change_from_2020
0,Alabama,AL,2020,ssp126,Low emissions (SSP126),33.073294,37.744867,37.744867,0.000000,0.341971,0.341971,0.000000
3,Alabama,AL,2021,ssp126,Low emissions (SSP126),40.702768,104.897632,37.744867,67.152766,45.387072,0.341971,45.045101
6,Alabama,AL,2022,ssp126,Low emissions (SSP126),33.566757,56.948924,37.744867,19.204058,0.812273,0.341971,0.470302
9,Alabama,AL,2023,ssp126,Low emissions (SSP126),35.915117,53.291383,37.744867,15.546516,6.165149,0.341971,5.823178
12,Alabama,AL,2024,ssp126,Low emissions (SSP126),38.485536,91.272590,37.744867,53.527724,20.854407,0.341971,20.512436


,state,state_abbr,year,season,scenario,scenario_label,summer_max_tasmax_c,summer_hot_days_30c,summer_hot_days_30c_2020,summer_hot_days_30c_change_from_2020,summer_hot_days_35c,summer_hot_days_35c_2020,summer_hot_days_35c_change_from_2020
0,Alabama,AL,2020,JJA,ssp126,Low emissions (SSP126),32.880560,27.418187,27.418187,0.000000,0.125239,0.125239,0.000000
3,Alabama,AL,2021,JJA,ssp126,Low emissions (SSP126),40.657354,77.890518,27.418187,50.472330,35.847606,0.125239,35.722367
6,Alabama,AL,2022,JJA,ssp126,Low emissions (SSP126),33.345303,43.328787,27.418187,15.910600,0.365165,0.125239,0.239926
9,Alabama,AL,2023,JJA,ssp126,Low emissions (SSP126),35.599708,44.718764,27.418187,17.300577,4.137506,0.125239,4.012267
12,Alabama,AL,2024,JJA,ssp126,Low emissions (SSP126),38.485536,66.203504,27.418187,38.785316,17.501189,0.125239,17.375950


,state,state_abbr,year,month,scenario,scenario_label,monthly_max_tasmax_c,monthly_hot_days_30c,monthly_hot_days_35c,monthly_hot_days_30c_2020_same_month,monthly_hot_days_30c_change_from_2020_same_month,monthly_hot_days_35c_2020_same_month,monthly_hot_days_35c_change_from_2020_same_month
0,Alabama,AL,2020,1,ssp126,Low emissions (SSP126),21.501943,0.000000,0.0,0.000000,0.0,0.0,0.0
1,Alabama,AL,2020,2,ssp126,Low emissions (SSP126),23.529976,0.000000,0.0,0.000000,0.0,0.0,0.0
2,Alabama,AL,2020,3,ssp126,Low emissions (SSP126),24.285777,0.000000,0.0,0.000000,0.0,0.0,0.0
3,Alabama,AL,2020,4,ssp126,Low emissions (SSP126),27.261599,0.000000,0.0,0.000000,0.0,0.0,0.0
4,Alabama,AL,2020,5,ssp126,Low emissions (SSP126),30.011487,1.334453,0.0,1.334453,0.0,0.0,0.0


In [13]:
county_heat_csv = OUTPUT_DIR / "county_annual_extreme_heat_cmip6_2020_2070.csv"
state_heat_csv = OUTPUT_DIR / "state_annual_extreme_heat_cmip6_2020_2070.csv"
state_summer_heat_csv = OUTPUT_DIR / "state_summer_heat_life_cmip6_2020_2070.csv"
state_monthly_heat_csv = OUTPUT_DIR / "state_monthly_heat_life_cmip6_2020_2070.csv"

county_heat.to_csv(county_heat_csv, index=False)
state_heat.to_csv(state_heat_csv, index=False)
state_summer_heat.to_csv(state_summer_heat_csv, index=False)
state_monthly_heat.to_csv(state_monthly_heat_csv, index=False)

print("Saved:")
print(county_heat_csv)
print(state_heat_csv)
print(state_summer_heat_csv)
print(state_monthly_heat_csv)

Saved:
/content/cmip6_heat_story_outputs/county_annual_extreme_heat_cmip6_2020_2070.csv
/content/cmip6_heat_story_outputs/state_annual_extreme_heat_cmip6_2020_2070.csv
/content/cmip6_heat_story_outputs/state_summer_heat_life_cmip6_2020_2070.csv
/content/cmip6_heat_story_outputs/state_monthly_heat_life_cmip6_2020_2070.csv


## 7. Export GeoJSON files for D3

In [14]:
states_export = states[["STATEFP", "STUSPS", "NAME", "geometry"]].copy()
states_export = states_export.rename(columns={
    "STUSPS": "state_abbr",
    "NAME": "state",
})

counties_export = county_meta[
    ["GEOID", "STATEFP", "COUNTYFP", "county", "state", "state_abbr", "geometry"]
].copy()

states_geojson = OUTPUT_DIR / "us_states.geojson"
counties_geojson = OUTPUT_DIR / "us_counties.geojson"

states_export.to_file(states_geojson, driver="GeoJSON")
counties_export.to_file(counties_geojson, driver="GeoJSON")

print("Saved:")
print(states_geojson)
print(counties_geojson)

Saved:
/content/cmip6_heat_story_outputs/us_states.geojson
/content/cmip6_heat_story_outputs/us_counties.geojson


## 8. Create combined files for easier D3 work

These files combine average temperature and hot-day metrics into one table for each time scale.

In [15]:
# Annual combined state file.
state_annual_combined = state_tas.merge(
    state_heat,
    on=["state", "state_abbr", "year", "scenario", "scenario_label"],
    how="left",
)

# Summer combined state file.
state_summer_combined = state_summer_tas.merge(
    state_summer_heat,
    on=["state", "state_abbr", "year", "season", "scenario", "scenario_label"],
    how="left",
)

# Monthly combined state file.
state_monthly_combined = state_monthly_tas.merge(
    state_monthly_heat,
    on=["state", "state_abbr", "year", "month", "scenario", "scenario_label"],
    how="left",
)

state_annual_combined_csv = OUTPUT_DIR / "state_annual_heat_life_combined_cmip6_2020_2070.csv"
state_summer_combined_csv = OUTPUT_DIR / "state_summer_heat_life_combined_cmip6_2020_2070.csv"
state_monthly_combined_csv = OUTPUT_DIR / "state_monthly_heat_life_combined_cmip6_2020_2070.csv"

state_annual_combined.to_csv(state_annual_combined_csv, index=False)
state_summer_combined.to_csv(state_summer_combined_csv, index=False)
state_monthly_combined.to_csv(state_monthly_combined_csv, index=False)

print("Saved combined files:")
print(state_annual_combined_csv)
print(state_summer_combined_csv)
print(state_monthly_combined_csv)

display(state_annual_combined.head())
display(state_summer_combined.head())
display(state_monthly_combined.head())

Saved combined files:
/content/cmip6_heat_story_outputs/state_annual_heat_life_combined_cmip6_2020_2070.csv
/content/cmip6_heat_story_outputs/state_summer_heat_life_combined_cmip6_2020_2070.csv
/content/cmip6_heat_story_outputs/state_monthly_heat_life_combined_cmip6_2020_2070.csv


,state,state_abbr,year,scenario,scenario_label,tas_c,tas_2020_c,tas_change_from_2020_c,annual_max_tasmax_c,hot_days_30c,hot_days_30c_2020,hot_days_30c_change_from_2020,hot_days_35c,hot_days_35c_2020,hot_days_35c_change_from_2020
0,Alabama,AL,2020,ssp126,Low emissions (SSP126),15.809565,15.809565,0.000000,33.073294,37.744867,37.744867,0.000000,0.341971,0.341971,0.000000
1,Alabama,AL,2021,ssp126,Low emissions (SSP126),16.903114,15.809565,1.093548,40.702768,104.897632,37.744867,67.152766,45.387072,0.341971,45.045101
2,Alabama,AL,2022,ssp126,Low emissions (SSP126),16.508223,15.809565,0.698658,33.566757,56.948924,37.744867,19.204058,0.812273,0.341971,0.470302
3,Alabama,AL,2023,ssp126,Low emissions (SSP126),16.367164,15.809565,0.557599,35.915117,53.291383,37.744867,15.546516,6.165149,0.341971,5.823178
4,Alabama,AL,2024,ssp126,Low emissions (SSP126),17.459909,15.809565,1.650343,38.485536,91.272590,37.744867,53.527724,20.854407,0.341971,20.512436


,state,state_abbr,year,season,scenario,scenario_label,summer_tas_c,summer_tas_c_2020,summer_tas_c_change_from_2020,summer_max_tasmax_c,summer_hot_days_30c,summer_hot_days_30c_2020,summer_hot_days_30c_change_from_2020,summer_hot_days_35c,summer_hot_days_35c_2020,summer_hot_days_35c_change_from_2020
0,Alabama,AL,2020,JJA,ssp126,Low emissions (SSP126),24.680512,24.680512,0.000000,32.880560,27.418187,27.418187,0.000000,0.125239,0.125239,0.000000
1,Alabama,AL,2021,JJA,ssp126,Low emissions (SSP126),28.238231,24.680512,3.557719,40.657354,77.890518,27.418187,50.472330,35.847606,0.125239,35.722367
2,Alabama,AL,2022,JJA,ssp126,Low emissions (SSP126),25.321384,24.680512,0.640872,33.345303,43.328787,27.418187,15.910600,0.365165,0.125239,0.239926
3,Alabama,AL,2023,JJA,ssp126,Low emissions (SSP126),25.333490,24.680512,0.652979,35.599708,44.718764,27.418187,17.300577,4.137506,0.125239,4.012267
4,Alabama,AL,2024,JJA,ssp126,Low emissions (SSP126),26.772844,24.680512,2.092333,38.485536,66.203504,27.418187,38.785316,17.501189,0.125239,17.375950


,state,state_abbr,year,month,scenario,scenario_label,monthly_tas_c,monthly_tas_c_2020_same_month,monthly_tas_c_change_from_2020_same_month,monthly_max_tasmax_c,monthly_hot_days_30c,monthly_hot_days_35c,monthly_hot_days_30c_2020_same_month,monthly_hot_days_30c_change_from_2020_same_month,monthly_hot_days_35c_2020_same_month,monthly_hot_days_35c_change_from_2020_same_month
0,Alabama,AL,2020,1,ssp126,Low emissions (SSP126),7.261339,7.261339,0.0,21.501943,0.000000,0.0,0.000000,0.0,0.0,0.0
1,Alabama,AL,2020,2,ssp126,Low emissions (SSP126),9.080862,9.080862,0.0,23.529976,0.000000,0.0,0.000000,0.0,0.0,0.0
2,Alabama,AL,2020,3,ssp126,Low emissions (SSP126),11.956863,11.956863,0.0,24.285777,0.000000,0.0,0.000000,0.0,0.0,0.0
3,Alabama,AL,2020,4,ssp126,Low emissions (SSP126),18.088153,18.088153,0.0,27.261599,0.000000,0.0,0.000000,0.0,0.0,0.0
4,Alabama,AL,2020,5,ssp126,Low emissions (SSP126),19.890474,19.890474,0.0,30.011487,1.334453,0.0,1.334453,0.0,0.0,0.0


In [16]:

# ============================================================
# Build final story files:
# observed 2000-2020 + CMIP6 projected 2020-2100
#
# Important:
# 2020 appears twice:
#   1. observed 2020
#   2. CMIP6 2020 under each scenario
#
# This lets us show:
#   - CMIP6 2020 minus observed 2020
#   - future change from observed 2020
#   - future change from CMIP6 2020
# ============================================================

# Add source label to projected CMIP6 files.
state_annual_combined["data_source"] = "projected"
state_summer_combined["data_source"] = "projected"
state_monthly_combined["data_source"] = "projected"

# -----------------------
# Annual story file
# -----------------------
annual_story_cols = [
    "state", "state_abbr", "year",
    "scenario", "scenario_label", "data_source",
    "tas_c",
    "annual_max_tasmax_c", "hot_days_30c", "hot_days_35c",
]

observed_annual_story = state_observed_annual_combined[annual_story_cols].copy()
projected_annual_story = state_annual_combined[annual_story_cols].copy()

state_annual_heat_story = pd.concat(
    [observed_annual_story, projected_annual_story],
    ignore_index=True,
)

for col in ["tas_c", "annual_max_tasmax_c", "hot_days_30c", "hot_days_35c"]:
    state_annual_heat_story = add_observed_and_cmip6_2020_deltas(
        df=state_annual_heat_story,
        observed_baseline_df=state_observed_annual_combined,
        value_col=col,
        id_cols=["state", "state_abbr"],
        monthly=False,
    )

state_annual_heat_story = state_annual_heat_story.sort_values(
    ["state", "data_source", "scenario", "year"]
)

# -----------------------
# Summer story file
# -----------------------
summer_story_cols = [
    "state", "state_abbr", "year", "season",
    "scenario", "scenario_label", "data_source",
    "summer_tas_c",
    "summer_max_tasmax_c", "summer_hot_days_30c", "summer_hot_days_35c",
]

observed_summer_story = state_observed_summer_combined[summer_story_cols].copy()
projected_summer_story = state_summer_combined[summer_story_cols].copy()

state_summer_heat_story = pd.concat(
    [observed_summer_story, projected_summer_story],
    ignore_index=True,
)

for col in ["summer_tas_c", "summer_max_tasmax_c", "summer_hot_days_30c", "summer_hot_days_35c"]:
    state_summer_heat_story = add_observed_and_cmip6_2020_deltas(
        df=state_summer_heat_story,
        observed_baseline_df=state_observed_summer_combined,
        value_col=col,
        id_cols=["state", "state_abbr"],
        monthly=False,
    )

state_summer_heat_story = state_summer_heat_story.sort_values(
    ["state", "data_source", "scenario", "year"]
)

# -----------------------
# Monthly story file
# -----------------------
monthly_story_cols = [
    "state", "state_abbr", "year", "month",
    "scenario", "scenario_label", "data_source",
    "monthly_tas_c",
    "monthly_max_tasmax_c", "monthly_hot_days_30c", "monthly_hot_days_35c",
]

observed_monthly_story = state_observed_monthly_combined[monthly_story_cols].copy()
projected_monthly_story = state_monthly_combined[monthly_story_cols].copy()

state_monthly_heat_story = pd.concat(
    [observed_monthly_story, projected_monthly_story],
    ignore_index=True,
)

for col in ["monthly_tas_c", "monthly_max_tasmax_c", "monthly_hot_days_30c", "monthly_hot_days_35c"]:
    state_monthly_heat_story = add_observed_and_cmip6_2020_deltas(
        df=state_monthly_heat_story,
        observed_baseline_df=state_observed_monthly_combined,
        value_col=col,
        id_cols=["state", "state_abbr"],
        monthly=True,
    )

state_monthly_heat_story = state_monthly_heat_story.sort_values(
    ["state", "data_source", "scenario", "year", "month"]
)

# -----------------------
# Save final story files
# -----------------------
annual_story_csv = OUTPUT_DIR / "state_annual_heat_story_observed_cmip6_2000_2100.csv"
summer_story_csv = OUTPUT_DIR / "state_summer_heat_story_observed_cmip6_2000_2100.csv"
monthly_story_csv = OUTPUT_DIR / "state_monthly_heat_story_observed_cmip6_2000_2100.csv"

state_annual_heat_story.to_csv(annual_story_csv, index=False)
state_summer_heat_story.to_csv(summer_story_csv, index=False)
state_monthly_heat_story.to_csv(monthly_story_csv, index=False)

print("Saved final story files:")
print(annual_story_csv)
print(summer_story_csv)
print(monthly_story_csv)

print("\nExample: summer story rows")
display(state_summer_heat_story.head())

print("\nCheck 2020 rows:")
display(
    state_summer_heat_story[
        (state_summer_heat_story["year"] == 2020)
        & (state_summer_heat_story["state"].isin(["California", "Texas", "Arizona"]))
    ][[
        "state", "year", "scenario", "scenario_label", "data_source",
        "summer_tas_c",
        "summer_tas_c_observed_2020",
        "summer_tas_c_cmip6_2020",
        "summer_tas_c_change_from_observed_2020",
        "summer_tas_c_change_from_cmip6_2020",
        "summer_tas_c_cmip6_2020_minus_observed_2020",
        "summer_hot_days_35c",
        "summer_hot_days_35c_observed_2020",
        "summer_hot_days_35c_cmip6_2020",
        "summer_hot_days_35c_change_from_observed_2020",
        "summer_hot_days_35c_change_from_cmip6_2020",
        "summer_hot_days_35c_cmip6_2020_minus_observed_2020",
    ]]
)


Saved final story files:
/content/cmip6_heat_story_outputs/state_annual_heat_story_observed_cmip6_2000_2100.csv
/content/cmip6_heat_story_outputs/state_summer_heat_story_observed_cmip6_2000_2100.csv
/content/cmip6_heat_story_outputs/state_monthly_heat_story_observed_cmip6_2000_2100.csv

Example: summer story rows


,state,state_abbr,year,season,scenario,scenario_label,data_source,summer_tas_c,summer_max_tasmax_c,summer_hot_days_30c,...,summer_hot_days_30c_observed_2020,summer_hot_days_30c_cmip6_2020,summer_hot_days_30c_change_from_observed_2020,summer_hot_days_30c_change_from_cmip6_2020,summer_hot_days_30c_cmip6_2020_minus_observed_2020,summer_hot_days_35c_observed_2020,summer_hot_days_35c_cmip6_2020,summer_hot_days_35c_change_from_observed_2020,summer_hot_days_35c_change_from_cmip6_2020,summer_hot_days_35c_cmip6_2020_minus_observed_2020
0,Alabama,AL,2000,JJA,observed,Observed,observed,26.768856,39.077120,84.880751,...,76.350901,NaN,8.529850,NaN,NaN,3.035902,NaN,27.445299,NaN,NaN
1,Alabama,AL,2001,JJA,observed,Observed,observed,25.558120,34.334240,69.370460,...,76.350901,NaN,-6.980440,NaN,NaN,3.035902,NaN,-2.959320,NaN,NaN
2,Alabama,AL,2002,JJA,observed,Observed,observed,26.156654,35.683935,82.693427,...,76.350901,NaN,6.342527,NaN,NaN,3.035902,NaN,1.126440,NaN,NaN
3,Alabama,AL,2003,JJA,observed,Observed,observed,25.491628,33.810311,66.422927,...,76.350901,NaN,-9.927973,NaN,NaN,3.035902,NaN,-3.035902,NaN,NaN
4,Alabama,AL,2004,JJA,observed,Observed,observed,25.262000,35.047462,61.200298,...,76.350901,NaN,-15.150603,NaN,NaN,3.035902,NaN,-2.002220,NaN,NaN



Check 2020 rows:


,state,year,scenario,scenario_label,data_source,summer_tas_c,summer_tas_c_observed_2020,summer_tas_c_cmip6_2020,summer_tas_c_change_from_observed_2020,summer_tas_c_change_from_cmip6_2020,summer_tas_c_cmip6_2020_minus_observed_2020,summer_hot_days_35c,summer_hot_days_35c_observed_2020,summer_hot_days_35c_cmip6_2020,summer_hot_days_35c_change_from_observed_2020,summer_hot_days_35c_change_from_cmip6_2020,summer_hot_days_35c_cmip6_2020_minus_observed_2020
41,Arizona,2020,observed,Observed,observed,27.318608,27.318608,NaN,0.000000,NaN,NaN,54.708678,54.708678,NaN,0.000000,NaN,NaN
1494,Arizona,2020,ssp126,Low emissions (SSP126),projected,26.934857,27.318608,26.934857,-0.383750,0.0,-0.383750,39.743026,54.708678,39.743026,-14.965652,0.0,-14.965652
1575,Arizona,2020,ssp245,Medium emissions (SSP245),projected,27.072136,27.318608,27.072136,-0.246472,0.0,-0.246472,37.084600,54.708678,37.084600,-17.624077,0.0,-17.624077
1656,Arizona,2020,ssp585,High emissions (SSP585),projected,27.400393,27.318608,27.400393,0.081786,0.0,0.081786,43.361076,54.708678,43.361076,-11.347602,0.0,-11.347602
83,California,2020,observed,Observed,observed,25.402479,25.402479,NaN,0.000000,NaN,NaN,41.973728,41.973728,NaN,0.000000,NaN,NaN
1980,California,2020,ssp126,Low emissions (SSP126),projected,23.373626,25.402479,23.373626,-2.028853,0.0,-2.028853,26.600306,41.973728,26.600306,-15.373422,0.0,-15.373422
2061,California,2020,ssp245,Medium emissions (SSP245),projected,22.958971,25.402479,22.958971,-2.443508,0.0,-2.443508,24.913464,41.973728,24.913464,-17.060264,0.0,-17.060264
2142,California,2020,ssp585,High emissions (SSP585),projected,22.934263,25.402479,22.934263,-2.468216,0.0,-2.468216,26.265530,41.973728,26.265530,-15.708198,0.0,-15.708198
860,Texas,2020,observed,Observed,observed,28.541301,28.541301,NaN,0.000000,NaN,NaN,50.485097,50.485097,NaN,0.000000,NaN,NaN
11214,Texas,2020,ssp126,Low emissions (SSP126),projected,27.284707,28.541301,27.284707,-1.256595,0.0,-1.256595,27.670870,50.485097,27.670870,-22.814227,0.0,-22.814227


## 9. Quick validation checks

Use this section to check whether the new story direction works before updating the webpage.

In [17]:
print("Files in output directory:")
for path in sorted(OUTPUT_DIR.glob("*")):
    print(path.name, "--", round(path.stat().st_size / 1_000_000, 2), "MB")

print("\nMissing values check:")
for name, df in [
    ("state_tas", state_tas),
    ("state_heat", state_heat),
    ("state_summer_combined", state_summer_combined),
    ("state_monthly_combined", state_monthly_combined),
]:
    print("\n", name)
    print(df.isna().sum().sort_values(ascending=False).head(12))

print("\nYear ranges:")
for name, df in [
    ("state_tas", state_tas),
    ("state_heat", state_heat),
    ("state_summer_combined", state_summer_combined),
    ("state_monthly_combined", state_monthly_combined),
]:
    print(name, df["year"].min(), df["year"].max(), "rows:", len(df))

print("\nExample: top states by additional 35°C+ annual hot days in 2070 under SSP585")
display(
    state_annual_combined[
        (state_annual_combined["year"] == 2070) &
        (state_annual_combined["scenario"] == "ssp585")
    ][
        ["state", "tas_change_from_2020_c", "hot_days_35c", "hot_days_35c_change_from_2020"]
    ]
    .sort_values("hot_days_35c_change_from_2020", ascending=False)
    .head(10)
)

print("\nExample: top states by additional summer 35°C+ hot days in 2070 under SSP585")
display(
    state_summer_combined[
        (state_summer_combined["year"] == 2070) &
        (state_summer_combined["scenario"] == "ssp585")
    ][
        ["state", "summer_tas_c_change_from_2020", "summer_hot_days_35c", "summer_hot_days_35c_change_from_2020"]
    ]
    .sort_values("summer_hot_days_35c_change_from_2020", ascending=False)
    .head(10)
)

print("\nExample monthly pattern: California, SSP585, 2070")
display(
    state_monthly_combined[
        (state_monthly_combined["state"] == "California") &
        (state_monthly_combined["scenario"] == "ssp585") &
        (state_monthly_combined["year"] == 2070)
    ][
        ["state", "year", "month", "monthly_tas_c", "monthly_hot_days_35c", "monthly_hot_days_35c_change_from_2020_same_month"]
    ]
)

Files in output directory:
county_annual_extreme_heat_cmip6_2020_2070.csv -- 75.16 MB
county_annual_tas_cmip6_2020_2070.csv -- 78.55 MB
observed_nclimgrid -- 0.02 MB
state_annual_extreme_heat_cmip6_2020_2070.csv -- 1.94 MB
state_annual_heat_life_combined_cmip6_2020_2070.csv -- 2.62 MB
state_annual_heat_story_observed_cmip6_2000_2100.csv -- 6.14 MB
state_annual_tas_cmip6_2020_2070.csv -- 1.27 MB
state_monthly_heat_life_cmip6_2020_2070.csv -- 17.27 MB
state_monthly_heat_life_combined_cmip6_2020_2070.csv -- 25.44 MB
state_monthly_heat_story_observed_cmip6_2000_2100.csv -- 59.77 MB
state_monthly_tas_cmip6_2020_2070.csv -- 15.61 MB
state_summer_heat_life_cmip6_2020_2070.csv -- 1.98 MB
state_summer_heat_life_combined_cmip6_2020_2070.csv -- 2.66 MB
state_summer_heat_story_observed_cmip6_2000_2100.csv -- 6.17 MB
state_summer_tas_cmip6_2020_2070.csv -- 1.32 MB
us_counties.geojson -- 2.27 MB
us_states.geojson -- 0.43 MB

Missing values check:

 state_tas
state                     0
state_abbr   

,state,tas_change_from_2020_c,hot_days_35c,hot_days_35c_change_from_2020
10418,Texas,0.340506,88.220158,22.471304
3857,Kansas,0.281898,43.284727,17.778799
2156,Florida,0.896219,21.136206,17.263239
8717,Oklahoma,-0.087466,62.984703,15.453562
6530,Nebraska,0.926923,21.038505,13.887058
212,Alabama,0.593411,13.723219,12.324531
2399,Georgia,0.697281,11.946825,6.998727
9932,South Dakota,0.780590,11.433288,6.822862
6044,Missouri,0.192833,11.073583,6.029043
4343,Louisiana,-0.152524,34.605822,5.578523



Example: top states by additional summer 35°C+ hot days in 2070 under SSP585


,state,summer_tas_c_change_from_2020,summer_hot_days_35c,summer_hot_days_35c_change_from_2020
6530,Nebraska,3.497938,20.303077,13.507075
10418,Texas,1.449668,61.716879,10.418241
2156,Florida,1.631198,11.478702,10.067652
212,Alabama,1.604329,9.564361,8.292105
3857,Kansas,1.473330,31.742528,8.116571
9932,South Dakota,3.012507,11.433288,6.822862
6044,Missouri,1.496983,10.551878,5.554986
698,Arizona,1.205139,48.555894,5.194818
6287,Montana,3.248870,6.849043,4.628106
8717,Oklahoma,0.864020,43.322780,4.347677



Example monthly pattern: California, SSP585, 2070


,state,year,month,monthly_tas_c,monthly_hot_days_35c,monthly_hot_days_35c_change_from_2020_same_month
101688,California,2070,1,6.611913,0.000000,0.000000
101689,California,2070,2,7.647553,0.000000,0.000000
101690,California,2070,3,9.100449,0.000000,0.000000
101691,California,2070,4,12.128748,0.000000,0.000000
101692,California,2070,5,16.410418,0.219136,-0.862928
101693,California,2070,6,19.883041,3.719580,-0.579164
101694,California,2070,7,23.911701,8.216510,-4.741604
101695,California,2070,8,25.790440,10.522717,1.514044
101696,California,2070,9,19.186897,1.849994,-6.343905
101697,California,2070,10,16.775840,0.000000,-2.883421


## 10. Zip and download all outputs

Run this after the validation looks reasonable.

In [19]:
import zipfile
from pathlib import Path
from google.colab import files

OUTPUT_DIR = Path("/content/cmip6_heat_story_outputs")

final_files = [
    "state_summer_heat_story_observed_cmip6_2000_2100.csv",
    "state_monthly_heat_story_observed_cmip6_2000_2100.csv",
    "state_annual_heat_story_observed_cmip6_2000_2100.csv",
    "us_states.geojson",
]

zip_path = "/content/cmip6_heat_story_final_site_data.zip"

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for filename in final_files:
        path = OUTPUT_DIR / filename
        if path.exists():
            z.write(path, arcname=filename)
        else:
            print("Missing:", path)

files.download(zip_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
import zipfile
from pathlib import Path
from google.colab import files

OUTPUT_DIR = Path("/content/cmip6_heat_story_outputs")
zip_path = "/content/cmip6_heat_story_processed_outputs_no_raw.zip"

# Exclude raw downloaded climate files and raw-data folders
exclude_dirs = {
    "observed_nclimgrid",
    "observed_cpc_tmax",
    "__MACOSX",
}

exclude_suffixes = {
    ".nc",
    ".zarr",
    ".idx",
    ".jsonl",
}

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in OUTPUT_DIR.rglob("*"):
        if path.is_dir():
            continue

        rel = path.relative_to(OUTPUT_DIR)

        # skip raw folders
        if any(part in exclude_dirs for part in rel.parts):
            continue

        # skip raw climate files
        if path.suffix.lower() in exclude_suffixes:
            continue

        # keep processed outputs: csv, geojson, json, txt, etc.
        z.write(path, arcname=str(rel))

print("Created processed-output zip:", zip_path)
files.download(zip_path)

Created processed-output zip: /content/cmip6_heat_story_processed_outputs_no_raw.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 11. Which files should go into the website `data/` folder?

For the observed + CMIP6 story, use these final story files:

- `state_summer_heat_story_observed_cmip6_2000_2100.csv` ← main story file
- `state_monthly_heat_story_observed_cmip6_2000_2100.csv` ← state detail / monthly breakdown
- `us_states.geojson` ← state map

Main map metrics:

- Average warming map: `summer_tas_c_change_from_observed_2020`
- Extreme heat map: `summer_hot_days_35c_change_from_observed_2020`
- Model-observation gap: `summer_hot_days_35c_cmip6_2020_minus_observed_2020`

Important interpretation:

- Observed average temperature comes from NOAA nClimGrid monthly `tavg`.
- Observed extreme hot days come from NOAA nClimGrid-Daily `tmax`.
- CMIP6 average temperature comes from monthly `tas`.
- CMIP6 extreme hot days come from daily `tasmax`.
- State values are area-weighted averages across counties, not total hot-day event counts.
- nClimGrid gridded observed data are CONUS, so observed rows are lower-48 states; Alaska and Hawaii projected rows will not have observed-baseline deltas unless a separate observed source is added for them.
